# O revisor do Nilo — treinar e medir

Treina um revisor pequeno e dedicado para o Andar 10 de *The Normal Elevator*, e
mede o resultado **com a mesma régua do jogo**.

**O que este caderno mede e o que ele NÃO mede.** Ele mede QUALIDADE: quantos
defeitos o modelo conserta, quantos ecos, cópias, promessas e quebras de cânone
ele comete. Ele **não** mede velocidade — TURNO só existe no navegador com o
wllama, e GPU de Colab não diz nada sobre o celular de ninguém. A carga continua
prevista pelo tamanho do arquivo (MB ÷ 32 ≈ segundos).

**Por que o julgamento roda em `node` e não em Python.** A régua mora em
`bancada-navegador/defeitos.mjs`, ao lado do cânone do jogo. Portar para Python
criaria uma segunda cópia para divergir — este projeto já pagou esse preço duas
vezes, quando a cópia da bancada ficou mais frouxa que o cânone do jogo e o
placar mentiu em dois modelos. Uma régua, um arquivo.

GPU: T4 (grátis) treina o 360M em poucos minutos; L4 usa bf16 e é mais rápida.


In [ ]:
!nvidia-smi -L || echo "SEM GPU — vá em Ambiente de execução › Alterar tipo › GPU"
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Dependências

`sentencepiece` não é usado pelo tokenizador do SmolLM2, mas o conversor do
llama.cpp tenta o caminho sentencepiece ANTES do BPE e só cai para o certo se o
import funcionar — sem ele a conversão morre com `ModuleNotFoundError`.

In [ ]:
!pip -q install "transformers>=4.44" peft accelerate sentencepiece gguf
# O julgamento roda em node. O Colab costuma já ter; se não tiver, instala.
!node --version || (apt-get -qq update && apt-get -qq install -y nodejs)

## 2. O repositório

O corpus, a prova e a régua vivem na branch de trabalho. Se o repositório for
privado, gere um token com escopo `repo` e cole em `TOKEN`.

In [ ]:
TOKEN = ""   # deixe vazio se o repositório for público
REPO  = "Felipe9272727/Jdjdjddj"
BRANCH = "claude/persistent-download-storage-i6l88v"

url = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
!rm -rf jdjdjddj && git clone -q --depth 1 -b {BRANCH} {url} jdjdjddj
%cd jdjdjddj/jubileu/bancada-navegador
!ls corpus

## 3. O corpus, conferido antes de treinar

`conferir.mjs` falha alto em dois casos, e os dois arruinam o treino em silêncio:

1. **treinar na prova** — se um caso da prova vazar para o corpus, o placar
   depois vira enfeite;
2. **treinar o defeito** — se uma frase "certa" quebrar o cânone, o modelo
   aprende a quebrá-lo com convicção.

In [ ]:
!node corpus/conferir.mjs && node corpus/gerar.mjs
!head -c 600 corpus/treino.jsonl

## 4. Treinar a v1 — o aluno que ainda não viu o professor

O aluno é o **Qwen3.5-0.8B**, e a escolha não foi por qualidade: é o **único
modelo pequeno que compartilha o vocabulário do professor de 27B**, e sem
vocabulário igual não existe KL nos logits mais adiante. Ele custa 834 MB contra
os 386 MB do SmolLM2 — 26 s de carga contra 12 s — e um terço disso é tabela de
embedding que o revisor nunca usa. A poda vem no fim.

`MESCLAR=0` salva só o adaptador. Para gerar (o passo seguinte) o PEFT carrega
base + LoRA e responde igual; mesclar é obrigatório só na hora do gguf.

Os alvos do LoRA são **descobertos**, não listados: 18 das 24 camadas deste
modelo são Gated DeltaNet e se chamam `in_proj_qkv/z/b/a` — com a lista fixa
antiga, três quartos do modelo ficariam sem adaptação.

In [ ]:
import os
os.environ["MODELO"]  = "Qwen/Qwen3.5-0.8B"
os.environ["SAIDA"]   = "corpus/revisor-v1"
os.environ["EPOCAS"]  = "8"
os.environ["LOTE"]    = "8"
os.environ["ACUMULA"] = "2"
os.environ["MESCLAR"] = "0"
os.environ["BF16"]    = "1"
!python3 corpus/treinar.py

## 5. Off-policy: o professor gera o corpus grande

Aqui mora o conserto do defeito real. O corpus escrito à mão tem **47 aberturas
distintas em 48 respostas** — variedade de forma não falta. O que falta é
**quantidade de casos**: 48 vistos 32 vezes cada viram decoreba, e o modelo passa
a encostar na resposta memorizada mais parecida.

O professor gera as duas pontas e a régua confere as duas: a frase errada TEM que
disparar a regra pedida, e o conserto não pode disparar nenhuma. O que o
professor erra é descartado em vez de virar treino.

O token é o do Hugging Face, com permissão de **Inference Providers**. O modelo
precisa do sufixo do provedor — sem ele o roteador responde
`model_not_supported`.

In [ ]:
from getpass import getpass
import os
os.environ["API_KEY"]  = getpass("token do Hugging Face (inference): ")
os.environ["API_URL"]  = "https://router.huggingface.co/v1/chat/completions"
os.environ["MODELO"]   = "Qwen/Qwen3.8-27B:featherless-ai"
os.environ["CASOS"]    = "300"     # comece com 10 para ver o custo
os.environ["POR_CASO"] = "4"       # K respostas por caso: é isto que espalha o repertório
os.environ["PENSAR"]   = "1"       # o professor pensa, e o alvo leva o <think>
!node corpus/destilar.mjs > corpus/destilado.jsonl
!wc -l corpus/destilado.jsonl

## 6. Treinar a v2 — agora com volume

Junta o corpus escrito à mão (a âncora de voz) com o destilado (o volume).

In [ ]:
!cat corpus/treino.jsonl corpus/destilado.jsonl > corpus/treino-v2.jsonl
!wc -l corpus/treino-v2.jsonl

import os
os.environ["MODELO"]  = "Qwen/Qwen3.5-0.8B"
os.environ["TREINO"]  = "corpus/treino-v2.jsonl"
os.environ["SAIDA"]   = "corpus/revisor-v2"
os.environ["EPOCAS"]  = "3"        # mais dados, menos épocas: é a decoreba que a gente está combatendo
os.environ["MESCLAR"] = "0"
!python3 corpus/treinar.py

## 7. On-policy: o aluno erra, e o professor corrige o erro DELE

O degrau que o relatório da Qwen chama de on-policy. Na destilação off-policy o
aluno só vê caminhos perfeitos e nunca aprende a sair de um buraco que ele mesmo
cava — e é exatamente esse o mecanismo da decoreba.

Esta é a versão **por sequência** (a troca é em texto). A versão forte alinha os
logits por KL e precisa do professor residente na GPU: 27,8B em 4 bits são
~15,5 GB, então **T4 não serve, L4 serve**. Como o aluno já compartilha o
vocabulário do professor, essa porta fica aberta para quando houver a placa.

In [ ]:
import json
# Os casos são os do corpus destilado, SEM a resposta: é o que o jogo manda ao
# revisor na hora da fala.
with open("corpus/casos.jsonl", "w") as saida:
    for linha in open("corpus/destilado.jsonl"):
        m = json.loads(linha)["messages"]
        saida.write(json.dumps({"messages": m[:-1]}, ensure_ascii=False) + "\n")

import os
os.environ["MODELO"]   = "corpus/revisor-v2"
os.environ["CASOS"]    = "corpus/casos.jsonl"
os.environ["POR_CASO"] = "2"
!python3 corpus/aluno-gera.py > corpus/aluno.jsonl
!wc -l corpus/aluno.jsonl

os.environ["MODELO"] = "Qwen/Qwen3.8-27B:featherless-ai"
os.environ["MODO"]   = "on"
!node corpus/destilar.mjs < corpus/aluno.jsonl > corpus/on-policy.jsonl
!wc -l corpus/on-policy.jsonl

## 7b. On-policy COMPLETO — KL de logits, professor residente

A célula 7 é on-policy **por sequência**: o professor reescreve a tentativa do aluno e o aluno treina no texto. Copia *o que* ele respondeu.

Esta aqui copia a **distribuição**: em cada posição, o quanto o professor achou de cada um dos 248 mil tokens. É onde mora a forma de pensar — o formato do raciocínio está nas probabilidades dos tokens dentro do `<think>`, não só no texto final.

**Precisa de A100.** O professor tem 27,78B: em bf16 são ~56 GB e não cabe em lugar nenhum do Colab; em 4 bits NF4 são ~16 GB e sobra espaço na A100 de 40 GB. Em L4 (24 GB) cabe apertado com `LOTE=1 GERA_TOKENS=80`, mas fica lento.

**Ordem certa:** rode a célula 6 (v2, off-policy) antes desta. Partir do zero aqui desperdiça a A100 aprendendo o que a SFT ensina de graça no L4.

O `FRESCOR=1` é o que faz ser on-policy de verdade: as sequências são regeradas pelo aluno **a cada passo**, com os pesos daquele instante.

In [ ]:
!pip -q install bitsandbytes
import os, torch

# A conta que decide o lote, feita na hora em vez de chutada: sobra depois do
# professor em 4 bits (~16 GiB) e do aluno com LoRA (~5 GiB).
total = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'GPU com {total:.0f} GiB')
grande = total >= 38

os.environ.update(
    MESTRE='Qwen/Qwen3.8-27B',
    BASE_ALUNO='Qwen/Qwen3.5-0.8B',
    ALUNO='corpus/revisor-v2',      # continua de onde a v2 parou
    SAIDA='corpus/revisor-v3',
    CORPUS='corpus/destilado.jsonl',
    BITS='4',
    PASSOS='600',
    FRESCOR='1',                    # regerar a cada passo = on-policy de verdade
    LOTE='2' if grande else '1',
    GERA_TOKENS='120' if grande else '80',
    FATIA='64' if grande else '32',
    T_KL='1.0',
    ALFA_CE='0.0',
    LR='1e-4',
)
!python3 corpus/on-policy-kl.py

## 8. Treinar a v3 — a versão que vai para a prova

In [ ]:
!cat corpus/treino.jsonl corpus/destilado.jsonl corpus/on-policy.jsonl > corpus/treino-v3.jsonl
!wc -l corpus/treino-v3.jsonl

import os
os.environ["MODELO"]  = "Qwen/Qwen3.5-0.8B"
os.environ["TREINO"]  = "corpus/treino-v3.jsonl"
os.environ["SAIDA"]   = "corpus/revisor-v3"
os.environ["EPOCAS"]  = "3"
os.environ["MESCLAR"] = "1"        # agora sim: o gguf precisa do modelo inteiro
!python3 corpus/treinar.py

## 5. A prova, e o julgamento com a régua do jogo

24 casos (os 6 históricos + 18 novos) e 3 controles — frases que já estavam
certas, onde a única falha possível é ESTRAGAR.

**Leia as saídas reprovadas.** A régua pega palavra proibida, eco, cópia,
fragmento e promessa. Ela não pega *"a few steps from a door that does not
exist"* — que é canônicamente errado (a porta existe, ela só não abre) e passa
liso. Modelo treinado erra em frase plausível, e é aí que a régua enxerga pior.

In [ ]:
!MODELO=corpus/revisor-v3 SAIDAS=corpus/saidas.jsonl python3 corpus/gerar-saidas.py
!node corpus/julgar-saidas.mjs corpus/saidas.jsonl

## 6. Para gguf, que é o que o jogo carrega

O conversor do llama.cpp deixou de ser um arquivo só: hoje `convert_hf_to_gguf.py`
importa o pacote `conversion/`, então baixar o script avulso não converte nada.
`--outtype q8_0` quantiza na conversão e evita compilar o `llama-quantize`.

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!LLAMACPP=/content/llama.cpp bash corpus/para-gguf.sh corpus/revisor-v3 revisor.gguf
from google.colab import files; files.download("revisor.gguf")

## 8. Publicar no Hugging Face (opcional)

O jogo carrega o revisor por URL, então o gguf precisa estar em algum lugar que
o navegador alcance. O token vai num campo desta célula e **não sai do seu
Colab** — nunca cole token de escrita em conversa, chat ou commit.

Crie um em huggingface.co/settings/tokens com permissão de **escrita**.

In [ ]:
from getpass import getpass
from huggingface_hub import HfApi

USUARIO = "Felipe0282829273"
NOME    = "nilo-revisor-v3"
token   = getpass("token de escrita do Hugging Face (não fica salvo): ")

api = HfApi(token=token)
repo = f"{USUARIO}/{NOME}"
api.create_repo(repo, repo_type="model", private=False, exist_ok=True)
api.upload_file(path_or_fileobj="revisor.gguf", path_in_repo="revisor-v3-q8_0.gguf", repo_id=repo)
print("URL para o jogo:")
print(f"https://huggingface.co/{repo}/resolve/main/revisor-v3-q8_0.gguf")

## 7. O que fazer com o arquivo

Suba o `revisor.gguf` para um repositório de modelos e aponte a entrada do
revisor em `src/npc/floor10Brains.ts` para a URL. A medição de TURNO (carga +
1ª chamada fria) é feita na bancada do navegador, não aqui:

```
PROVA=grande RODADAS=2 ENUNCIADO=treinado SISTEMA=treinado TEMPERATURA=0 \
  MODELOS="revisor.gguf:REVISOR:q8_0:1024" node revisor-candidatos.mjs
```
